# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the [FAIR^2 dataset package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant Schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access toplevel metadata fields safely:
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'n/a')}")
print(f"Version: {getattr(dataset.metadata, 'version', 'n/a')}")


## 2. Data Overview
List the available record sets, their IDs, associated fields and columns and their `@id`s.

**Note:** All dataset entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# List all record sets and their fields
record_set_infos = []
for recset in dataset.metadata.record_sets:
    print(f"Record Set: {getattr(recset, 'name', '')} | @id: {recset.id}")
    field_ids = []
    field_names = []
    for field in getattr(recset, 'fields', []):
        print(f"  Field:   {getattr(field, 'name', '')} | @id: {field.id} | Data type: {getattr(field, 'data_type', '')}")
        field_ids.append(field.id)
        field_names.append(getattr(field, 'name', ''))
    col_ids = []
    if getattr(recset, 'columns', None):
        for col in recset.columns:
            print(f"  Column:  {getattr(col, 'name', '')} | @id: {col.id}")
            col_ids.append(col.id)
    record_set_infos.append({'name': getattr(recset, 'name', ''), 'id': recset.id, 'fields': field_ids, 'columns': col_ids})
    print('-' * 60)

print(f"Found {len(dataset.metadata.record_sets)} record sets.")

## 3. Data Extraction
Load all record sets as pandas DataFrames. Record sets and fields are referenced by their Croissant `@id`.

In [ ]:
# Extract data from available record sets
# We'll first build a list of all record set @ids
record_sets_ids = [recset.id for recset in dataset.metadata.record_sets]
dataframes = {}

for recset_id in record_sets_ids:
    print(f"Loading records for: {recset_id}")
    try:
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load data for {recset_id}: {e}")

# For further analysis, pick the first record set as main example
if len(record_sets_ids) > 0:
    main_recset_id = record_sets_ids[0]
    print(f"\nMain record set ID: {main_recset_id}")
    print(f"Available columns: {dataframes[main_recset_id].columns.tolist()}\n")

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and optionally grouping. Provide examples relevant to the available numeric fields.

- Remove records where a numeric value is too low.
- Normalize the numeric field (z-score).
- Optionally, group by a categorical field and compute summary statistics.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with the actual field `@id`s from the chosen record set. All references are by `@id`.

In [ ]:
# Selecting suitable numeric and group fields from the main record set
df = dataframes[main_recset_id]

# List all columns (these are field @ids)
print(f"Available fields (@id): {df.columns.tolist()}")

# Attempt to infer a numeric field: pick first with float/integer dtype or known
# You may adapt the known field ids below based on Data Overview output above
numeric_candidates = [
    c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])
]
if len(numeric_candidates) == 0:
    # Try to coerce numeric columns by searching likely field names
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    
if len(numeric_candidates) == 0:
    raise Exception("No numeric field found.")

numeric_field_id = numeric_candidates[0]
print(f"Using numeric field @id for analysis: {numeric_field_id}")

# Find a group-able field (guess as a non-numeric field)
possible_groups = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
if possible_groups:
    group_field_id = possible_groups[0]
    print(f"Using group field @id for grouping: {group_field_id}")
else:
    group_field_id = None

# Filtering: threshold for numeric field
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered to {len(filtered_df)} records with {numeric_field_id} > {threshold}")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally group
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Basic visualizations of a numeric field distribution and summaries (histogram, boxplot, group means if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram
plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of numeric field: {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.tight_layout()
plt.show()

# Boxplot, possibly grouped
plt.figure(figsize=(8, 4))
if group_field_id and group_field_id in df.columns:
    # Limit number of groups for clarity
    top_groups = df[group_field_id].value_counts().head(10).index
    subdf = df[df[group_field_id].isin(top_groups)]
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=subdf)
    plt.title(f"{numeric_field_id} by {group_field_id} (top 10 groups)")
    plt.xticks(rotation=45)
else:
    sns.boxplot(y=df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id}")
plt.tight_layout()
plt.show()


## 6. Conclusion
- We loaded the dataset metadata and explored all record sets using `mlcroissant`.
- We extracted the data, examined available fields, and performed basic preprocessing.
- We filtered, normalized, and visualized numeric values by referencing all dataset entities by their Croissant `@id`.

Proceed to more advanced analysis or modeling as required for your research application.